# 01 — Data Exploration

This notebook explores hourly electricity price data across 8 North American ISOs
(Independent System Operators). We'll understand what LMP (Locational Marginal Pricing)
looks like, compare price dynamics across markets, and identify key patterns.

## What is an ISO?
ISOs are regional organizations that coordinate electricity generation and transmission.
Each ISO operates its own wholesale electricity market with unique characteristics:
- **ERCOT** (Texas): isolated grid, high wind penetration, extreme summer peaks
- **PJM** (Mid-Atlantic): largest US market, coal/gas/nuclear mix
- **CAISO** (California): high solar, famous "duck curve"
- **MISO** (Midwest): wind corridor, agricultural demand patterns
- **NYISO** (New York): congested transmission, high urban demand
- **ISO-NE** (New England): gas-dependent, winter price spikes
- **SPP** (Southwest): wind-rich, lowest average prices
- **IESO** (Ontario): hydro/nuclear baseload, moderate prices

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import ISODataFetcher
from src.data.weather import WeatherFetcher
from src.data.processor import DataProcessor

In [ ]:
# Fetch data for all ISOs (uses synthetic fallback if APIs unavailable)
fetcher = ISODataFetcher(config_path='../config/settings.yaml')

end_date = pd.Timestamp.now().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.now() - pd.Timedelta(days=365)).strftime('%Y-%m-%d')

all_prices = fetcher.fetch_all(start_date, end_date)
print(f"Total rows: {len(all_prices):,}")
print(f"ISOs: {all_prices['iso'].unique()}")
all_prices.head()

In [ ]:
# Basic statistics per ISO
summary = all_prices.groupby('iso')['lmp'].agg(
    ['mean', 'std', 'min', 'max', 'median', 'count']
).round(2)
summary['cv'] = (summary['std'] / summary['mean'] * 100).round(1)
summary.sort_values('mean', ascending=False)

In [ ]:
# Price time series by ISO
daily = all_prices.copy()
daily['date'] = daily['timestamp'].dt.date
daily_avg = daily.groupby(['date', 'iso'])['lmp'].mean().reset_index()

fig = px.line(daily_avg, x='date', y='lmp', color='iso',
              title='Daily Average LMP by ISO',
              labels={'lmp': 'LMP ($/MWh)', 'date': 'Date'})
fig.update_layout(height=500)
fig.show()

In [ ]:
# Price distribution by ISO (box plot)
fig = px.box(all_prices, x='iso', y='lmp',
             title='LMP Distribution by ISO',
             labels={'lmp': 'LMP ($/MWh)'})
fig.update_layout(height=500)
fig.show()

In [ ]:
# Hourly shape (average price by hour of day)
all_prices['hour'] = all_prices['timestamp'].dt.hour
hourly_avg = all_prices.groupby(['hour', 'iso'])['lmp'].mean().reset_index()

fig = px.line(hourly_avg, x='hour', y='lmp', color='iso',
              title='Average Hourly Price Shape by ISO',
              labels={'lmp': 'Avg LMP ($/MWh)', 'hour': 'Hour of Day'})
fig.update_layout(height=500)
fig.show()

## Key Observations

1. **CAISO duck curve**: Notice the midday price depression (solar generation) and evening ramp
2. **ERCOT volatility**: Highest price variance due to isolated grid and weather sensitivity
3. **SPP/IESO**: Lowest prices due to wind generation and hydro baseload respectively
4. **Peak hours (7-22)**: All ISOs show higher prices during daytime demand hours
5. **Cross-ISO spreads**: Different price levels create spread trading opportunities